In [1]:
# Cell 1 — Loading, splitting and patch creation
import os
import glob
import h5py
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tqdm import tqdm
from Helpers import extract_metadata

# --- Configuration ---
BASE_DIR = "data/GC_data"
TEST_SUBDIR = "20250702"
DMIN_KM = 19.5
DMAX_KM = 22.0
SAMPLE_STEP = 1        # temporal downsample factor
CHANNEL_STEP = 1       # keep every channel in selected km window

# Patch params (must match training)
TIME_PATCH_STEPS = 128
DISTANCE_PATCH_STEPS = 128
PATCH_OVERLAP = 0.1

# Discover files
all_files = sorted(glob.glob(os.path.join(BASE_DIR, "**", "*.hdf5"), recursive=True))
test_files = sorted(glob.glob(os.path.join(BASE_DIR, TEST_SUBDIR, "*.hdf5")))
# train_files = all hdf5 under BASE_DIR that are NOT inside the TEST_SUBDIR folder
train_files = [p for p in all_files if TEST_SUBDIR not in os.path.relpath(p, BASE_DIR).split(os.path.sep)]

print(f"Found {len(train_files)} training files and {len(test_files)} test files.")

if not train_files:
    raise RuntimeError("No training files found under data/GC_data — cannot proceed.")

# Reference file to build distance axis (prefer training file)
ref_file = train_files[0] if train_files else test_files[0]
_, dt, dx, channels_info, _ = extract_metadata(ref_file)

# Normalize channels_info to an indices array
if np.isscalar(channels_info):
    channel_count = int(channels_info)
    channel_indices = np.arange(channel_count)
else:
    channel_indices = np.asarray(channels_info)

distance_array_km = channel_indices * dx / 1000.0
dist_mask = (distance_array_km >= DMIN_KM) & (distance_array_km <= DMAX_KM)
selected_count = int(np.sum(dist_mask))
if selected_count == 0:
    raise RuntimeError(f"No channels found in km range {DMIN_KM}-{DMAX_KM}.")

print(f"Selected km window {DMIN_KM}-{DMAX_KM} km -> {selected_count} channels")

def load_and_combine(paths, sample_step=1, channel_step=1):
    arrays = []
    for p in tqdm(paths, desc="Loading files"):
        with h5py.File(p, "r") as f:
            # dataset assumed to be 'data' with shape (time, channels)
            data = f['data'][::sample_step, dist_mask][:, ::channel_step]
            arrays.append(data)
    if not arrays:
        raise RuntimeError("No arrays loaded — check file list and dataset name 'data'.")
    combined = np.vstack(arrays) if len(arrays) > 1 else arrays[0]
    return combined

# Load training and fit scaler (no leakage)
combined_train = load_and_combine(train_files, sample_step=SAMPLE_STEP, channel_step=CHANNEL_STEP)
print("Combined train shape:", combined_train.shape)

scaler = MinMaxScaler()
train_flat = combined_train.astype(np.float32).ravel().reshape(-1, 1)
scaler.fit(train_flat)
normalized_train = scaler.transform(train_flat).reshape(combined_train.shape)

# Load test (if any) and normalize using the same scaler
if test_files:
    combined_test = load_and_combine(test_files, sample_step=SAMPLE_STEP, channel_step=CHANNEL_STEP)
    test_flat = combined_test.astype(np.float32).ravel().reshape(-1, 1)
    normalized_test = scaler.transform(test_flat).reshape(combined_test.shape)
    print("Combined test shape:", normalized_test.shape)
else:
    normalized_test = None
    print("No test files found in test subdir.")

# Patch extraction (produces patches and optional simulated labels for test)
def create_patches_and_labels(data_matrix, time_window, dist_window, overlap_factor, is_test_set=False):
    patches = []
    labels = []
    time_steps, dist_channels = data_matrix.shape
    time_step_size = max(1, int(time_window * (1 - overlap_factor)))
    dist_step_size = max(1, int(dist_window * (1 - overlap_factor)))
    for t in range(0, time_steps - time_window + 1, time_step_size):
        for d in range(0, dist_channels - dist_window + 1, dist_step_size):
            patch = data_matrix[t:t + time_window, d:d + dist_window]
            if patch.shape == (time_window, dist_window):
                patches.append(patch)
                if is_test_set:
                    # simple simulated label rule: unusually large pixel => anomaly (adjust threshold if needed)
                    labels.append(1 if np.max(patch) > 0.8 else 0)
                else:
                    labels.append(0)
    if patches:
        patches = np.array(patches)
        labels = np.array(labels)
    else:
        patches = np.empty((0, time_window, dist_window), dtype=np.float32)
        labels = np.empty((0,), dtype=int)
    return patches, labels

# Generate patches
X_train_raw, _ = create_patches_and_labels(normalized_train, TIME_PATCH_STEPS, DISTANCE_PATCH_STEPS, PATCH_OVERLAP, is_test_set=False)
if normalized_test is not None:
    X_test_raw, y_test_simulated = create_patches_and_labels(normalized_test, TIME_PATCH_STEPS, DISTANCE_PATCH_STEPS, PATCH_OVERLAP, is_test_set=True)
else:
    X_test_raw, y_test_simulated = np.empty((0, TIME_PATCH_STEPS, DISTANCE_PATCH_STEPS)), np.empty((0,), dtype=int)

# Add channel dimension (channels=1) for CNN
X_train = X_train_raw[..., np.newaxis].astype(np.float32)
X_test = X_test_raw[..., np.newaxis].astype(np.float32)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_test_simulated shape:", y_test_simulated.shape)

Helpers.py loaded successfully.
Found 115 training files and 341 test files.
Selected km window 19.5-22.0 km -> 1224 channels


Loading files: 100%|██████████| 115/115 [01:07<00:00,  1.70it/s]


Combined train shape: (718750, 1224)


Loading files: 100%|██████████| 341/341 [03:28<00:00,  1.64it/s]


Combined test shape: (2131250, 1224)


MemoryError: Unable to allocate 3.81 GiB for an array with shape (62490, 128, 128, 1) and data type float32

In [ ]:
# Cell 2 — Model definition and training (Convolutional Autoencoder)
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D
from tensorflow.keras.models import Model

# Ensure preprocessed data is present
if 'X_train' not in globals():
    raise RuntimeError("X_train not found — run the loading cell first.")

input_shape = X_train.shape[1:]  # (time, dist, 1)

def build_cae(input_shape):
    input_img = Input(shape=input_shape)
    # encoder
    x = Conv2D(32, (3, 3), activation='relu', padding='same')(input_img)
    x = MaxPooling2D((2, 2), padding='same')(x)
    x = Conv2D(16, (3, 3), activation='relu', padding='same')(x)
    x = MaxPooling2D((2, 2), padding='same')(x)
    # latent
    encoded = Conv2D(8, (3, 3), activation='relu', padding='same', name='encoded')(x)
    # decoder
    x = Conv2D(16, (3, 3), activation='relu', padding='same')(encoded)
    x = UpSampling2D((2, 2))(x)
    x = Conv2D(32, (3, 3), activation='relu', padding='same')(x)
    x = UpSampling2D((2, 2))(x)
    decoded = Conv2D(1, (3, 3), activation='sigmoid', padding='same')(x)
    model = Model(input_img, decoded)
    return model

cae_model = build_cae(input_shape)
cae_model.compile(optimizer='adam', loss='mse')
cae_model.summary()

# Training params
BATCH_SIZE = 32
EPOCHS = 15

# Train using .fit (validation_split uses a portion of the training patches)
history = cae_model.fit(
    X_train, X_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.1,
    shuffle=True
)

# Save model for later reuse
cae_model.save("cae_model.h5")
print("Saved model to cae_model.h5")

In [ ]:
# Cell 3 — Evaluation: reconstruction errors, thresholding and visualization
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model

# Ensure test data exists
if 'X_test' not in globals():
    raise RuntimeError("X_test not found — run loading cell first.")

# Load model if needed
if 'cae_model' not in globals():
    cae_model = load_model("cae_model.h5")

# Predict reconstructions for train and test
X_train_pred = cae_model.predict(X_train, verbose=0)
train_error = np.mean(np.square(X_train - X_train_pred), axis=(1, 2, 3))

if len(X_test) > 0:
    X_test_pred = cae_model.predict(X_test, verbose=0)
    test_error = np.mean(np.square(X_test - X_test_pred), axis=(1, 2, 3))
else:
    X_test_pred = np.empty((0,))
    test_error = np.empty((0,))

# Threshold from training distribution (robust percentile)
PERCENTILE = 99.9
anomaly_threshold = np.percentile(train_error, PERCENTILE)

print(f"Train errors: mean={train_error.mean():.6e}, std={train_error.std():.6e}")
print(f"Anomaly threshold (train {PERCENTILE}th percentile): {anomaly_threshold:.6e}")

# If simulated labels are available compute metrics
if 'y_test_simulated' in globals() and len(y_test_simulated) == len(test_error) and len(test_error) > 0:
    y_pred_binary = (test_error > anomaly_threshold).astype(int)
    precision = precision_score(y_test_simulated, y_pred_binary, zero_division=0)
    recall = recall_score(y_test_simulated, y_pred_binary, zero_division=0)
    f1 = f1_score(y_test_simulated, y_pred_binary, zero_division=0)
    accuracy = accuracy_score(y_test_simulated, y_pred_binary)
    print("\nEvaluation metrics (using simulated labels):")
    print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}, Accuracy: {accuracy:.4f}")

# Show top anomalous patches and plot top one
if len(test_error) > 0:
    top_k = min(5, len(test_error))
    top_idx = np.argsort(test_error)[::-1][:top_k]
    print(f"\nTop {top_k} anomalous patch indices (in X_test):")
    for rank, idx in enumerate(top_idx, 1):
        label = int(y_test_simulated[idx]) if 'y_test_simulated' in globals() and idx < len(y_test_simulated) else None
        print(f"{rank}. idx={idx}, score={test_error[idx]:.6e}, simulated_label={label}")

    # Plot the top anomalous patch (original vs reconstruction)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    best = top_idx[0]
    orig = X_test[best].squeeze()
    recon = X_test_pred[best].squeeze()
    vmax = max(orig.max(), recon.max())
    axes[0].imshow(orig, aspect='auto', cmap='viridis', vmax=vmax)
    axes[0].set_title('Original (top anomalous)')
    axes[1].imshow(recon, aspect='auto', cmap='viridis', vmax=vmax)
    axes[1].set_title('Reconstruction')
    plt.tight_layout()
    plt.show()
else:
    print("No test patches to evaluate.")